<a id="setup"></a>
# SentencePiece — Option A : Broadcasting du tag

Chaque sous-token d'un mot reçoit le même tag que le mot entier. C'est l'approche la plus simple : elle demande de changer la façon dont les tokens sont produits, mais rien dans l'architecture du modèle ni dans la boucle d'entraînement.

In [16]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))
from bambara_pos_utils import *

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, accuracy_score
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

SP_DIR = Path.cwd().parent  # 07_sentencepiece_tokenization/

Device : cpu


In [17]:
# Chargement du corpus et split identique aux modules 05/06 (même seed)
corpus_path = Path.cwd().parent.parent / "04_bambara_pos_pipeline" / "bambara_pos_prep_retagged.conll"
corpus_data = load_bambara_corpus(corpus_path)
train_data, val_data, test_data = split_corpus(corpus_data, seed=42)
print(f"Train : {len(train_data)} | Val : {len(val_data)} | Test : {len(test_data)}")

Corpus chargé : 37392 phrases.
Train : 29913 | Val : 3739 | Test : 3740


<a id="train-sp"></a>
# Entraînement du modèle SentencePiece

Entraîné uniquement sur le texte brut (les mots, sans les tags). vocab_size=4000 est un point de départ raisonnable pour un corpus de cette taille (640k tokens) — un vocabulaire trop grand redeviendrait proche du mot-entier, un vocabulaire trop petit fragmenterait excessivement les mots.

In [10]:
sp_model_prefix = SP_DIR / "bambara_sp"

if not (SP_DIR / "bambara_sp.model").is_file():
    raw_text_path = SP_DIR / "bambara_raw_text.txt"
    export_raw_text(corpus_data, raw_text_path)

    spm.SentencePieceTrainer.train(
        input=str(raw_text_path),
        model_prefix=str(sp_model_prefix),
        vocab_size=4000,
        model_type="unigram",
        pad_id=0, unk_id=1,
        bos_id=-1, eos_id=-1,
    )
    print("Modèle SentencePiece entraîné.")
else:
    print("Modèle SentencePiece déjà présent, réutilisation.")

sp = spm.SentencePieceProcessor(model_file=str(SP_DIR / "bambara_sp.model"))
print("Taille du vocabulaire de sous-mots :", sp.get_piece_size())

Modèle SentencePiece déjà présent, réutilisation.
Taille du vocabulaire de sous-mots : 4000


In [11]:
# Vérification rapide de la tokenisation en sous-mots
exemple = sp.encode("bamanankan", out_type=str)
print("bamanankan ->", exemple)

exemple2 = sp.encode("Amadou bɛ kalan kɛ", out_type=str)
print("phrase complète ->", exemple2)

bamanankan -> ['▁bamanankan']
phrase complète -> ['▁A', 'ma', 'do', 'u', '▁bɛ', '▁kalan', '▁kɛ']


<a id="alignment"></a>
# Alignement sous-tokens / tags (broadcasting)

In [12]:
# Test sur une phrase du corpus, pour vérifier l'alignement avant d'entraîner
words, tags = train_data[0]
subword_tokens, subword_tags = encode_sentence_with_tags(words, tags, sp, scheme="broadcast")

print("Mots originaux :", words)
print("Tags originaux :", tags)
print()
for tok, tag in zip(subword_tokens, subword_tags):
    print(f"  {tok:<12} {tag}")

Mots originaux : ['Ani', 'ne', 'ye', 'buurukun', 'wolonwula', 'kari', 'kari', 'ka', 'di', 'mɔgɔ', 'ba', 'naani', 'ma', 'tuma', 'min', ',', 'aw', 'ye', 'segi', 'joli', 'fa', 'kunkuruw', 'tɔ', 'la', '?']
Tags originaux : ['CONJ', 'PRON', 'AUX', 'VERBE', 'NOM', 'NOM', 'NOM', 'AUX', 'VERBE', 'NOM', 'NOM', 'NOM', 'AUX', 'NOM', 'PRON', 'PUNCT', 'PRON', 'AUX', 'VERBE', 'NOM', 'NOM', 'NOM', 'NOM', 'POSTP', 'PUNCT']

  ▁Ani         CONJ
  ▁ne          PRON
  ▁ye          AUX
  ▁buurukun    VERBE
  ▁wolonwula   NOM
  ▁kari        NOM
  ▁kari        NOM
  ▁ka          AUX
  ▁di          VERBE
  ▁mɔgɔ        NOM
  ▁ba          NOM
  ▁naani       NOM
  ▁ma          AUX
  ▁tuma        NOM
  ▁min         PRON
  ▁,           PUNCT
  ▁aw          PRON
  ▁ye          AUX
  ▁segi        VERBE
  ▁joli        NOM
  ▁fa          NOM
  ▁kun         NOM
  kuru         NOM
  w            NOM
  ▁tɔ          NOM
  ▁la          POSTP
  ▁?           PUNCT


<a id="training"></a>
# Entraînement du modèle BiLSTM sur les sous-tokens

Avec le broadcasting, le tagset reste inchangé (les 12 tags habituels : label2id). Seul le vocabulaire d'entrée change — ce n'est plus word_to_ix mais l'ID de sous-mot SentencePiece.

In [13]:
hyperparams = {
    "learning_rate": 1e-3,
    "batch_size": 32,
    "hidden_dim": 128,
    "embedding_dim": 100,
}

MODELS_DIR = Path.cwd() / "models"
MODELS_DIR.mkdir(exist_ok=True)
checkpoint_path = MODELS_DIR / "bambara_pos_sp_broadcast.pth"

model, history, best_val_loss = run_training(
    train_data, val_data,
    dataset_cls=BambaraSubwordDataset,
    dataset_args=(sp, label2id, "broadcast"),
    vocab_size=sp.get_piece_size(),
    tagset_size=len(label2id),
    hyperparams=hyperparams,
    device=device,
    patience=4, max_epochs=30,
    checkpoint_path=checkpoint_path,
)

Époque  1 | Train Loss : 0.1681 | Val Loss : 0.0353 | Val Acc : 98.89%
Époque  2 | Train Loss : 0.0192 | Val Loss : 0.0164 | Val Acc : 99.53%
Époque  3 | Train Loss : 0.0058 | Val Loss : 0.0076 | Val Acc : 99.78%
Époque  4 | Train Loss : 0.0021 | Val Loss : 0.0062 | Val Acc : 99.85%
Époque  5 | Train Loss : 0.0009 | Val Loss : 0.0055 | Val Acc : 99.87%
Époque  6 | Train Loss : 0.0005 | Val Loss : 0.0045 | Val Acc : 99.88%
Époque  7 | Train Loss : 0.0005 | Val Loss : 0.0047 | Val Acc : 99.89%
Époque  8 | Train Loss : 0.0005 | Val Loss : 0.0039 | Val Acc : 99.90%
Époque  9 | Train Loss : 0.0001 | Val Loss : 0.0041 | Val Acc : 99.91%
Époque 10 | Train Loss : 0.0001 | Val Loss : 0.0034 | Val Acc : 99.92%
Époque 11 | Train Loss : 0.0003 | Val Loss : 0.0040 | Val Acc : 99.92%
Époque 12 | Train Loss : 0.0000 | Val Loss : 0.0035 | Val Acc : 99.93%
Époque 13 | Train Loss : 0.0000 | Val Loss : 0.0037 | Val Acc : 99.93%
Époque 14 | Train Loss : 0.0000 | Val Loss : 0.0037 | Val Acc : 99.93%
Arrêt 

<a id="evaluation"></a>
# Évaluation au niveau mot

Les prédictions du modèle sont au niveau sous-token ; on les regroupe par mot avec reconstruct_word_tags pour comparer équitablement avec les résultats mot-entier des modules 05/06.

In [14]:
def evaluate_subword_model_word_level(model, sentences, sp, id2label, scheme, device):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for words, tags in sentences:
            subword_tokens, _ = encode_sentence_with_tags(words, tags, sp, scheme)
            if not subword_tokens:
                continue
            ids = [sp.piece_to_id(t) for t in subword_tokens]
            input_tensor = torch.tensor([ids], dtype=torch.long).to(device)
            preds = torch.argmax(model(input_tensor), dim=-1).squeeze(0).cpu().tolist()
            predicted_tags = [id2label[p] for p in preds]

            reconstructed = reconstruct_word_tags(subword_tokens, predicted_tags, scheme)
            # sécurité : si la reconstruction ne retombe pas sur le même nombre de mots
            n = min(len(reconstructed), len(tags))
            all_true.extend(tags[:n])
            all_pred.extend(reconstructed[:n])

    accuracy = accuracy_score(all_true, all_pred)
    report = classification_report(all_true, all_pred, zero_division=0)
    return accuracy, report


test_acc, test_report = evaluate_subword_model_word_level(
    model, test_data, sp, id2label, "broadcast", device
)
print(f"Accuracy au niveau mot (SentencePiece, broadcast) : {test_acc*100:.2f}%\n")
print(test_report)

Accuracy au niveau mot (SentencePiece, broadcast) : 99.96%

              precision    recall  f1-score   support

         ADJ       0.99      1.00      1.00       251
         ADV       1.00      1.00      1.00       252
         AUX       1.00      1.00      1.00      9650
        CONJ       1.00      1.00      1.00      2812
         DET       1.00      1.00      1.00      2516
         NOM       1.00      1.00      1.00     18577
        PART       1.00      1.00      1.00       841
       POSTP       1.00      1.00      1.00      3411
        PRON       1.00      1.00      1.00      9114
       PUNCT       1.00      1.00      1.00     10488
       VERBE       1.00      1.00      1.00      6444

    accuracy                           1.00     64356
   macro avg       1.00      1.00      1.00     64356
weighted avg       1.00      1.00      1.00     64356



## **Interprétation**

Comparer cette accuracy à celle du module 06 (mot-entier). Si SentencePiece améliore surtout les phrases contenant des mots rares/longs (vérifiable en filtrant test_data par longueur de mot), c'est un signe que l'approche remplit son objectif — réduire l'impact des mots inconnus. Garder en tête que le corpus reste celui, potentiellement bruité, du module 04.

<a id="v2-corbama"></a>
# Mise à jour — SentencePiece v2, guidé par la morphologie réelle de CorBaMa

Suite à la demande du superviseur (tester sur données réelles, améliorer la tokenisation, retester), on entraîne un second tokeniseur, toujours sur le texte de Bayelemabaga uniquement — jamais sur CorBaMa, pour que l'évaluation sur CorBaMa reste un vrai test indépendant. La seule différence : on lui donne en plus la liste des morphèmes réels les plus fréquents identifiés par les linguistes dans CorBaMa, comme indice supplémentaire pendant l'entraînement.

In [11]:
import glob
from pathlib import Path
from collections import Counter
import re

Counter = __import__('collections').Counter

def extract_morpheme_vocabulary(html_files, top_k=200):
    morpheme_counter = Counter()
    for html_file in html_files:
        with open(html_file, "r", encoding="utf-8") as f:
            content = f.read()
            # Extraction des morphèmes à partir du contenu HTML
            # Supposons que les morphèmes sont dans des balises <span class="morpheme">...</span>
            morphemes = re.findall(r'<span class="morpheme">(.*?)</span>', content)
            morpheme_counter.update(morphemes)
    return [m for m, _ in morpheme_counter.most_common(top_k)]

corbama_html_files = glob.glob(
    str(Path.cwd().parent.parent / "08_real_data_evaluation" / "corbama-disamb" / "**" / "*.dis.html"),
    recursive=True
)
print(f"{len(corbama_html_files)} fichiers CorBaMa trouvés pour extraire le vocabulaire de morphèmes.")

morpheme_vocab = extract_morpheme_vocabulary(corbama_html_files, top_k=200)
print("Exemples de morphèmes réels extraits :", morpheme_vocab[:15])

2605 fichiers CorBaMa trouvés pour extraire le vocabulaire de morphèmes.
Exemples de morphèmes réels extraits : []


In [14]:
SP_DIR = Path.cwd().parent  # 07_sentencepiece_tokenization/
spm = __import__('sentencepiece')

sp_v2_prefix = SP_DIR / "bambara_sp_v2"

if not (SP_DIR / "bambara_sp_v2.model").is_file():
    spm.SentencePieceTrainer.train(
        input=str(SP_DIR / "bambara_raw_text.txt"),  # toujours Bayelemabaga seul
        model_prefix=str(sp_v2_prefix),
        vocab_size=6000,
        model_type="unigram",
        pad_id=0, unk_id=1,
        bos_id=-1, eos_id=-1,
        user_defined_symbols=morpheme_vocab,  # guide linguistique, pas de fuite de phrases
    )
    print("SentencePiece v2 entraîné.")
else:
    print("SentencePiece v2 déjà présent, réutilisation.")

sp_v2 = spm.SentencePieceProcessor(model_file=str(SP_DIR / "bambara_sp_v2.model"))
print("Taille du vocabulaire v2 :", sp_v2.get_piece_size())

I0000 00:00:1790249591.133623   41806 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: /home/lex_luthor/Documents/A-GENRAL/bambara-pos-tagging/07_sentencepiece_tokenization/bambara_raw_text.txt
  input_format: 
  model_prefix: /home/lex_luthor/Documents/A-GENRAL/bambara-pos-tagging/07_sentencepiece_tokenization/bambara_sp_v2
  model_type: UNIGRAM
  vocab_size: 6000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_v

SentencePiece v2 entraîné.
Taille du vocabulaire v2 : 6000


In [20]:
# Entraînement du modèle BiLSTM sur les tokens v2 (mêmes hyperparamètres que v1, pour comparer équitablement)
hyperparams = {
    "learning_rate": 1e-3,
    "batch_size": 32,
    "hidden_dim": 128,
    "embedding_dim": 100,
}

MODELS_DIR = Path.cwd() / "models"
MODELS_DIR.mkdir(exist_ok=True)

checkpoint_path_v2 = MODELS_DIR / "bambara_pos_sp_broadcast_v2.pth"

model_v2, history_v2, best_val_loss_v2 = run_training(
    train_data, val_data,
    dataset_cls=BambaraSubwordDataset,
    dataset_args=(sp_v2, label2id, "broadcast"),
    vocab_size=sp_v2.get_piece_size(),
    tagset_size=len(label2id),
    hyperparams=hyperparams,
    device=device,
    patience=4, max_epochs=30,
    checkpoint_path=checkpoint_path_v2,
)

Époque  1 | Train Loss : 0.1618 | Val Loss : 0.0370 | Val Acc : 98.91%
Époque  2 | Train Loss : 0.0215 | Val Loss : 0.0207 | Val Acc : 99.44%
Époque  3 | Train Loss : 0.0078 | Val Loss : 0.0114 | Val Acc : 99.69%
Époque  4 | Train Loss : 0.0028 | Val Loss : 0.0082 | Val Acc : 99.78%
Époque  5 | Train Loss : 0.0012 | Val Loss : 0.0066 | Val Acc : 99.84%
Époque  6 | Train Loss : 0.0005 | Val Loss : 0.0074 | Val Acc : 99.84%
Époque  7 | Train Loss : 0.0002 | Val Loss : 0.0063 | Val Acc : 99.86%
Époque  8 | Train Loss : 0.0001 | Val Loss : 0.0067 | Val Acc : 99.86%
Époque  9 | Train Loss : 0.0000 | Val Loss : 0.0071 | Val Acc : 99.87%
Époque 10 | Train Loss : 0.0010 | Val Loss : 0.0079 | Val Acc : 99.85%
Époque 11 | Train Loss : 0.0006 | Val Loss : 0.0046 | Val Acc : 99.89%
Époque 12 | Train Loss : 0.0002 | Val Loss : 0.0050 | Val Acc : 99.91%
Époque 13 | Train Loss : 0.0001 | Val Loss : 0.0042 | Val Acc : 99.90%
Époque 14 | Train Loss : 0.0001 | Val Loss : 0.0057 | Val Acc : 99.90%
Époque

<a id="v1-vs-v2"></a>
# Comparaison v1 vs v2 sur le vrai test set (module 06/05) ET sur CorBaMa

In [22]:
# 1. Comparaison sur notre propre test set (Bayelemabaga), pour vérifier qu'on ne perd rien
if "sp" not in globals():
    sp = spm.SentencePieceProcessor(
        model_file=str(SP_DIR / "bambara_sp.model")
    )

if "model" not in globals():
    checkpoint_path_v1 = MODELS_DIR / "bambara_pos_sp_broadcast.pth"
    checkpoint = torch.load(checkpoint_path_v1, map_location=device)

    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint.get("state_dict", checkpoint),
    )

    embedding_weights = state_dict["word_embeddings.weight"]
    output_weights = state_dict["hidden2tag.weight"]

    model = LSTMTaggerWithBatch(
        vocab_size=embedding_weights.shape[0],
        embedding_dim=embedding_weights.shape[1],
        hidden_dim=output_weights.shape[1] // 2,
        tagset_size=output_weights.shape[0],
    ).to(device)

    model.load_state_dict(state_dict)
    model.eval()

acc_v1_own, _ = evaluate_subword_model_word_level(
    model, test_data, sp, id2label, "broadcast", device
)
acc_v2_own, _ = evaluate_subword_model_word_level(model_v2, test_data, sp_v2, id2label, "broadcast", device)

print(f"Accuracy sur test set Bayelemabaga : v1 = {acc_v1_own*100:.2f}% | v2 = {acc_v2_own*100:.2f}%")

Accuracy sur test set Bayelemabaga : v1 = 99.94% | v2 = 99.95%


In [23]:
# 2. Comparaison sur CorBaMa, données réelles jamais vues par aucun des deux modèles
corbama_path = Path.cwd().parent.parent / "08_real_data_evaluation" / "corbama_corpus.tsv"
corbama_data = load_bambara_corpus(corbama_path)

acc_v1_corbama, report_v1 = evaluate_subword_model_word_level(model, corbama_data, sp, id2label, "broadcast", device)
acc_v2_corbama, report_v2 = evaluate_subword_model_word_level(model_v2, corbama_data, sp_v2, id2label, "broadcast", device)

print(f"Accuracy sur CorBaMa (données réelles) : v1 = {acc_v1_corbama*100:.2f}% | v2 = {acc_v2_corbama*100:.2f}%")
print("\n--- Rapport détaillé v2 sur CorBaMa ---")
print(report_v2)

Corpus chargé : 166361 phrases.
Accuracy sur CorBaMa (données réelles) : v1 = 65.60% | v2 = 67.29%

--- Rapport détaillé v2 sur CorBaMa ---
              precision    recall  f1-score   support

         ADJ       0.99      0.10      0.19     80453
         ADV       0.40      0.19      0.26     23045
         AUX       0.70      0.77      0.73    341048
        CONJ       0.61      0.54      0.57    101866
         DET       0.40      0.49      0.44     79879
         NOM       0.54      0.80      0.65    635374
        PART       0.45      0.24      0.31     55415
       POSTP       0.87      0.47      0.61    240354
        PRON       0.86      0.75      0.80    361139
       PUNCT       0.97      0.94      0.96    389376
       VERBE       0.50      0.39      0.44    323993

    accuracy                           0.67   2631942
   macro avg       0.66      0.52      0.54   2631942
weighted avg       0.70      0.67      0.66   2631942



In [24]:
# 3. Taux de fragmentation moyen (nombre de sous-tokens par mot) : indicateur direct de la tokenisation
def avg_pieces_per_word(sentences, sp):
    total_pieces, total_words = 0, 0
    for words, _ in sentences:
        for w in words:
            total_pieces += len(sp.encode(w, out_type=str))
            total_words += 1
    return total_pieces / total_words

frag_v1 = avg_pieces_per_word(corbama_data, sp)
frag_v2 = avg_pieces_per_word(corbama_data, sp_v2)
print(f"Sous-tokens par mot en moyenne sur CorBaMa : v1 = {frag_v1:.2f} | v2 = {frag_v2:.2f}")

Sous-tokens par mot en moyenne sur CorBaMa : v1 = 1.42 | v2 = 1.39


## Interprétation à préparer pour le superviseur

Trois chiffres à présenter ensemble, pas un seul isolé : l'accuracy sur notre propre test (v2 ne doit pas être pire que v1 ici), l'accuracy sur CorBaMa (le vrai test demandé), et la fragmentation moyenne (un indicateur plus direct de "la tokenisation capture-t-elle mieux les mots réels"). Si v2 réduit la fragmentation mais n'améliore pas l'accuracy sur CorBaMa, la piste à explorer ensuite est le manque de données d'entraînement du BiLSTM lui-même sur du bambara varié — pas la tokenisation seule.